In [1]:
import pandas as pd

df = pd.read_csv("enron_spam_data.csv")

print(df.head())

   Message ID                       Subject  \
0           0  christmas tree farm pictures   
1           1      vastar resources , inc .   
2           2  calpine daily gas nomination   
3           3                    re : issue   
4           4     meter 7268 nov allocation   

                                             Message Spam/Ham        Date  
0                                                NaN      ham  1999-12-10  
1  gary , production from the high island larger ...      ham  1999-12-13  
2             - calpine daily gas nomination 1 . doc      ham  1999-12-14  
3  fyi - see note below - already done .\nstella\...      ham  1999-12-14  
4  fyi .\n- - - - - - - - - - - - - - - - - - - -...      ham  1999-12-14  


In [2]:
print(df.columns)

Index(['Message ID', 'Subject', 'Message', 'Spam/Ham', 'Date'], dtype='object')


In [3]:
print(len(df))

33716


In [4]:
df.head(5)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [5]:
# handle missing values
df = df.dropna(subset=['Message'])

In [6]:
# combine subject and message into a single text column
df['text'] = df['Subject'] + ' ' + df['Message']

In [7]:
# rename label column
df = df.rename(columns={'Spam/Ham': 'label'})

df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [8]:
# Keep only the relevant columns
df = df[['text', 'label']]

In [9]:
# verify the changes
print(df.head())

                                                text  label
1  vastar resources , inc . gary , production fro...      0
2  calpine daily gas nomination - calpine daily g...      0
3  re : issue fyi - see note below - already done...      0
4  meter 7268 nov allocation fyi .\n- - - - - - -...      0
5  mcmullen gas for 11 / 99 jackie ,\nsince the i...      0


In [10]:
print(df['label'].value_counts())

label
1    16852
0    16493
Name: count, dtype: int64


In [11]:
print(df.isnull().sum())

text     238
label      0
dtype: int64


In [12]:
df = df.dropna(subset=['text']) 

In [13]:
df = df[df['text'].str.strip() != ""]  # Remove rows where 'text' is empty or contains only whitespace

In [14]:
print(df.isnull().sum())
print(len(df))

text     0
label    0
dtype: int64
33107


In [15]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # lowercase
    text = text.lower()
    
    # remove numbers
    text = re.sub(r'\d+', '', text) # remove digits
    
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text) # remove punctuation except for word characters and whitespace
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip() # remove leading/trailing whitespace and reduce multiple spaces to single
    
    # remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kotha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
df['clean_text'] = df['text'].apply(clean_text)

In [17]:
print("Before:\n", df['text'].iloc[0])
print("\nAfter:\n", df['clean_text'].iloc[0])

Before:
 vastar resources , inc . gary , production from the high island larger block a - 1 # 2 commenced on
saturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and
10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .
george x 3 - 6992
- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16
am - - - - - - - - - - - - - - - - - - - - - - - - - - -
daren j farmer
12 / 10 / 99 10 : 38 am
to : carlos j rodriguez / hou / ect @ ect
cc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect
subject : vastar resources , inc .
carlos ,
please call linda and get everything set up .
i ' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each
following day based on my conversations with bill fischer at bmar .
d .
- - - - - - - - - - - - - - - - - - - - - - forwarded by daren j farmer / hou / ect on 12 / 10 / 99 10 : 34
am - - - - - - - - - - - - - - - - - - -

In [18]:
custom_noise = {
    'subject', 'fw', 'fwd', 're', 'cc',
    'ect', 'hou', 'corp', 'forwarded'
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    words = [
        word for word in words 
        if word not in stop_words and word not in custom_noise
    ]
    
    return " ".join(words)

In [19]:
df['clean_text'] = df['text'].apply(clean_text)

In [20]:
print("Before:\n", df['text'].iloc[0])
print("\nAfter:\n", df['clean_text'].iloc[0])

Before:
 vastar resources , inc . gary , production from the high island larger block a - 1 # 2 commenced on
saturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and
10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .
george x 3 - 6992
- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16
am - - - - - - - - - - - - - - - - - - - - - - - - - - -
daren j farmer
12 / 10 / 99 10 : 38 am
to : carlos j rodriguez / hou / ect @ ect
cc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect
subject : vastar resources , inc .
carlos ,
please call linda and get everything set up .
i ' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each
following day based on my conversations with bill fischer at bmar .
d .
- - - - - - - - - - - - - - - - - - - - - - forwarded by daren j farmer / hou / ect on 12 / 10 / 99 10 : 34
am - - - - - - - - - - - - - - - - - - -

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=7000, ngram_range=(1,2), min_df=5)

X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

In [22]:
print(X.shape)

(33107, 7000)


In [23]:
print(X[0])

  (0, 5255)	0.14388748254808362
  (0, 2941)	0.125989485734548
  (0, 2509)	0.08490338972524487
  (0, 4839)	0.10882382918523949
  (0, 2764)	0.06078079955946404
  (0, 3153)	0.10414367784041798
  (0, 3331)	0.09456832742687234
  (0, 632)	0.09905012902077344
  (0, 5450)	0.04400310464305723
  (0, 2655)	0.15209868114818867
  (0, 833)	0.15002367581805126
  (0, 2144)	0.07983170910754267
  (0, 6336)	0.1489570134929352
  (0, 4358)	0.04860247573928116
  (0, 2550)	0.2954622890265396
  (0, 6794)	0.29001500567495614
  (0, 1446)	0.10910299294696317
  (0, 2231)	0.11942453012335093
  (0, 5368)	0.05135690552461336
  (0, 3796)	0.0905368793612347
  (0, 2643)	0.10361852315934603
  (0, 4614)	0.032732692118594336
  (0, 785)	0.026206535133293442
  (0, 3447)	0.24921878365660363
  (0, 2557)	0.04169525830191002
  :	:
  (0, 5527)	0.02373593096958785
  (0, 5796)	0.03749485772733209
  (0, 1262)	0.04696390887616235
  (0, 132)	0.04355848502507562
  (0, 6023)	0.04243528188893808
  (0, 2488)	0.028268587838433726
  (0, 41

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
print(X_train.shape)
print(X_test.shape)

(26485, 7000)
(6622, 7000)


In [26]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [27]:
y_pred = model.predict(X_test)

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9833887043189369
Precision: 0.9807980798079808
Recall: 0.9861236802413273
F1 Score: 0.983453670276775


In [29]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[3243   64]
 [  46 3269]]


In [30]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [31]:
y_pred_lr = lr_model.predict(X_test)

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

Accuracy: 0.9868619752340683
Precision: 0.9815035799522673
Recall: 0.9924585218702866
F1 Score: 0.9869506524673767


In [33]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test,  y_pred_lr))

[[3245   62]
 [  25 3290]]


In [34]:
y_probs = lr_model.predict_proba(X_test)[:, 1]

In [35]:
import numpy as np

threshold = 0.6
y_pred_custom = (y_probs >= threshold).astype(int)

In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall:", recall_score(y_test, y_pred_custom))
print("F1:", f1_score(y_test, y_pred_custom))
print(confusion_matrix(y_test, y_pred_custom))

Precision: 0.988821752265861
Recall: 0.9873303167420815
F1: 0.9880754716981132
[[3270   37]
 [  42 3273]]


In [37]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
coefficients = lr_model.coef_[0]

top_spam_idx = np.argsort(coefficients)[-10:]
top_ham_idx = np.argsort(coefficients)[:10]

print("Top Spam Words:")
for i in reversed(top_spam_idx):
    print(feature_names[i], coefficients[i])

print("\nTop Ham Words:")
for i in top_ham_idx:
    print(feature_names[i], coefficients[i])

Top Spam Words:
http 4.15126663363511
software 3.675178238765009
remove 3.6401400759382407
money 3.5242232370536573
life 3.3996556571196495
account 3.262888831405332
click 3.153075050244718
online 3.143563869095422
info 2.9908826681240868
de 2.8144633456896733

Top Ham Words:
enron -11.67368785702425
vince -6.3949926839714
thanks -6.044360642564133
attached -5.345724614896344
louise -5.26806725034964
questions -3.8063136017397063
pm -3.693559704186114
enron com -3.613792279344098
let know -3.6131580961065852
houston -3.5453244209765704


In [38]:
features = vectorizer.get_feature_names_out()

print(features[:60])

['aa' 'ab' 'abacha' 'abiiity' 'ability' 'able' 'abroad' 'absence'
 'absolute' 'absolutely' 'abuse' 'abuse program' 'ac' 'academic' 'accept'
 'acceptable' 'acceptance' 'acceptance company' 'acceptance terms'
 'accepted' 'accepting' 'access' 'access request' 'accessories' 'accident'
 'accomplish' 'accomplishments' 'accordance' 'according' 'accordingly'
 'account' 'account information' 'account manager' 'accountant'
 'accounting' 'accounting investment' 'accounts' 'accuracy'
 'accuracy completeness' 'accurate' 'achieve' 'achieved' 'achievements'
 'acquire' 'acquired' 'acquiring' 'acquisition' 'acquisitions' 'acrobat'
 'acrobat professional' 'across' 'act' 'act amended' 'act section'
 'act statements' 'action' 'action may' 'actions' 'actions may' 'active']


In [39]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.9833887043189369
Precision: 0.9807980798079808
Recall: 0.9861236802413273
F1: 0.983453670276775


In [40]:
top_spam_idx = np.argsort(lr_model.coef_[0])[-10:]

for i in reversed(top_spam_idx):
    print(feature_names[i])

http
software
remove
money
life
account
click
online
info
de


In [41]:
import numpy as np

def extract_metadata(text):
    return [
        text.count("http"),  # link count
        sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0,  # uppercase ratio
        len(text),  # length
        sum(text.count(c) for c in ['!', '$', '%'])  # special chars
    ]

meta_features = np.array(df['text'].apply(extract_metadata).tolist())

In [42]:
from scipy.sparse import hstack

X_combined = hstack([X, meta_features])

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)  

In [44]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

c:\Users\kotha\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [45]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

Accuracy: 0.986408939897312
Precision: 0.9809126155681479
Recall: 0.9921568627450981
F1 Score: 0.9865026994601079


In [46]:
y_probs = lr_model.predict_proba(X_test)[:, 1]

threshold = 0.6  # start here
y_pred_custom = (y_probs >= threshold).astype(int)

In [47]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_custom))
print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall:", recall_score(y_test, y_pred_custom))
print("F1 Score:", f1_score(y_test, y_pred_custom))

Accuracy: 0.987012987012987
Precision: 0.9893907244619582
Recall: 0.9846153846153847
F1 Score: 0.9869972785001512


In [48]:
THRESHOLD = 0.6

y_probs = lr_model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= THRESHOLD).astype(int)

In [49]:
import pandas as pd

sms_df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)

In [50]:
sms_df['label'] = sms_df['label'].map({'ham': 0, 'spam': 1})

In [51]:
sms_df['clean_text'] = sms_df['text'].apply(clean_text)

In [52]:
print(df.columns)

Index(['text', 'label', 'clean_text'], dtype='object')


In [53]:
print(sms_df.columns)

Index(['label', 'text', 'clean_text'], dtype='object')


In [54]:
sms_df = sms_df[['clean_text', 'label']]
df = df[['clean_text', 'label']]

In [55]:
df_combined = pd.concat([df, sms_df], ignore_index=True)

In [56]:
print(df_combined.shape)
print(df_combined['label'].value_counts())

(38679, 2)
label
0    21318
1    17361
Name: count, dtype: int64


In [62]:
vectorizer = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,2),
    min_df=5
)

X = vectorizer.fit_transform(df_combined['clean_text'])
y = df_combined['label']

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [66]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

In [67]:
print(len(y_test))
print(len(y_pred))

7736
7736


In [68]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9689762150982419
Precision: 0.9738173162749182
Recall: 0.9556204379562043
F1 Score: 0.964633068081344


In [69]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[4223   88]
 [ 152 3273]]


In [70]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      4311
           1       0.97      0.96      0.96      3425

    accuracy                           0.97      7736
   macro avg       0.97      0.97      0.97      7736
weighted avg       0.97      0.97      0.97      7736



In [71]:
y_probs = lr_model.predict_proba(X_test)[:, 1]

In [72]:
threshold = 0.6
y_pred_custom = (y_probs >= threshold).astype(int)

In [73]:
print("Custom Threshold Results:")
print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall:", recall_score(y_test, y_pred_custom))
print("F1:", f1_score(y_test, y_pred_custom))
print(confusion_matrix(y_test, y_pred_custom))

Custom Threshold Results:
Precision: 0.9846342962507683
Recall: 0.9354744525547445
F1: 0.9594250636322803
[[4261   50]
 [ 221 3204]]


In [74]:
import pandas as pd

df = pd.DataFrame(columns=["text", "predicted", "actual"])
df.to_csv("data/feedback.csv", index=False)